# 🎙️ Accent Classifier — Full Training Pipeline

**Model:** `facebook/wav2vec2-base` fine-tuned with a classification head  
**Dataset:** Mozilla Common Voice 13.0 (English)  
**Accents:** American · British · Indian · Australian · Canadian  

Run this notebook on **Kaggle (P100 GPU)** or **Google Colab (T4 GPU)**.  
Expected training time: **2–3 hours** for 5 epochs.

---

## 0 — Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate gradio librosa torchaudio accelerate soundfile

## 1 — Configuration

In [ ]:
# ─── Config ────────────────────────────────────────────────
BASE_MODEL = "facebook/wav2vec2-base"
SAMPLING_RATE = 16_000
MAX_LENGTH_SAMPLES = 48_000  # 3 seconds

ACCENT_LABELS = ["us", "england", "indian", "australia", "canada"]
LABEL2ID = {l: i for i, l in enumerate(ACCENT_LABELS)}
ID2LABEL = {i: l for i, l in enumerate(ACCENT_LABELS)}
NUM_LABELS = len(ACCENT_LABELS)

DISPLAY_LABELS = [
    "🇺🇸 American", "🇬🇧 British", "🇮🇳 Indian",
    "🇦🇺 Australian", "🇨🇦 Canadian"
]

OUTPUT_DIR = "accent-classifier"
FINAL_MODEL_DIR = "accent-classifier-final"

# Set to smaller numbers for a quick test run
MAX_TRAIN_SAMPLES = None  # e.g. 500 for testing
MAX_EVAL_SAMPLES = None   # e.g. 100 for testing

## 2 — Load & Filter Dataset

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

ds = DatasetDict()
for split in ["train", "validation", "test"]:
    print(f"Loading {split}...")
    ds[split] = load_dataset(
        "mozilla-foundation/common_voice_13_0", "en",
        split=split, trust_remote_code=True
    )

# Filter to target accents
for split in ds:
    before = len(ds[split])
    ds[split] = ds[split].filter(
        lambda batch: [a in ACCENT_LABELS for a in batch["accent"]],
        batched=True, batch_size=1000
    )
    print(f"  {split}: {before} → {len(ds[split])} samples")

# Optional: cap samples for quick testing
if MAX_TRAIN_SAMPLES:
    ds["train"] = ds["train"].select(range(min(MAX_TRAIN_SAMPLES, len(ds["train"]))))
if MAX_EVAL_SAMPLES:
    ds["validation"] = ds["validation"].select(range(min(MAX_EVAL_SAMPLES, len(ds["validation"]))))
    ds["test"] = ds["test"].select(range(min(MAX_EVAL_SAMPLES, len(ds["test"]))))

## 3 — Preprocess (Resample + Extract Features)

In [ ]:
from transformers import Wav2Vec2FeatureExtractor

# Resample to 16 kHz
ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

# Load feature extractor
extractor = Wav2Vec2FeatureExtractor.from_pretrained(BASE_MODEL)

def preprocess(batch):
    audio = [x["array"] for x in batch["audio"]]
    inputs = extractor(
        audio, sampling_rate=SAMPLING_RATE,
        max_length=MAX_LENGTH_SAMPLES,
        truncation=True, padding=True
    )
    inputs["labels"] = [LABEL2ID[a] for a in batch["accent"]]
    return inputs

print("Preprocessing... (this may take a while)")
ds = ds.map(
    preprocess, batched=True, batch_size=32,
    remove_columns=ds["train"].column_names, num_proc=1
)
ds.set_format("torch")
print("Done! ✅")
for split in ds:
    print(f"  {split}: {len(ds[split])} samples")

## 4 — Build Model

In [ ]:
from transformers import Wav2Vec2ForSequenceClassification

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)

# Freeze CNN feature encoder — only train transformer + classification head
model.freeze_feature_encoder()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")
print(f"Frozen params:    {total - trainable:,}")

## 5 — Train

In [ ]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_ratio=0.1,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    push_to_hub=False,
    save_total_limit=2,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
)

print("🚀 Starting training...")
trainer.train()

## 6 — Evaluate & Save

In [ ]:
# Evaluate on validation set
metrics = trainer.evaluate()
print(f"\n📊 Validation Accuracy: {metrics['eval_accuracy']:.4f}")

# Evaluate on test set
test_metrics = trainer.evaluate(ds["test"])
print(f"📊 Test Accuracy:       {test_metrics['eval_accuracy']:.4f}")

# Save the final model
trainer.save_model(FINAL_MODEL_DIR)
print(f"\n✅ Model saved to '{FINAL_MODEL_DIR}'")

## 7 — Test Inference (Optional)

In [ ]:
import torch

# Load saved model for inference test
inference_model = Wav2Vec2ForSequenceClassification.from_pretrained(FINAL_MODEL_DIR)
inference_model.eval()

# Take a sample from the test set (before preprocessing)
test_raw = load_dataset(
    "mozilla-foundation/common_voice_13_0", "en",
    split="test", trust_remote_code=True
)
test_raw = test_raw.filter(lambda x: x["accent"] in ACCENT_LABELS)
test_raw = test_raw.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

sample = test_raw[0]
print(f"True accent: {sample['accent']}")

inputs = extractor(
    sample["audio"]["array"],
    sampling_rate=SAMPLING_RATE,
    return_tensors="pt", padding=True
)

with torch.no_grad():
    logits = inference_model(**inputs).logits

probs = torch.softmax(logits, dim=-1)[0]
for i, label in enumerate(DISPLAY_LABELS):
    print(f"  {label}: {probs[i]:.4f}")

predicted = DISPLAY_LABELS[torch.argmax(probs).item()]
print(f"\nPredicted: {predicted}")

## 8 — Gradio Demo (Run in Colab/Kaggle)

Run the cell below to get a live mic-based accent classifier.

In [ ]:
import gradio as gr
import librosa

demo_model = Wav2Vec2ForSequenceClassification.from_pretrained(FINAL_MODEL_DIR)
demo_model.eval()

def classify_accent(audio_path):
    if audio_path is None:
        return {label: 0.0 for label in DISPLAY_LABELS}
    audio, sr = librosa.load(audio_path, sr=SAMPLING_RATE)
    if len(audio) < SAMPLING_RATE * 0.5:
        return {label: 0.0 for label in DISPLAY_LABELS}
    inputs = extractor(audio, sampling_rate=SAMPLING_RATE, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = demo_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    return {DISPLAY_LABELS[i]: float(probs[i]) for i in range(NUM_LABELS)}

demo = gr.Interface(
    fn=classify_accent,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label="🎤 Speak for 3–5 seconds"),
    outputs=gr.Label(num_top_classes=5, label="🌍 Accent Prediction"),
    title="🎙️ Accent Classifier",
    description="Speak into your mic and find out which English accent you have.",
)

demo.launch(share=True, debug=True)

## 9 — Download Model (Optional)

Download the trained model to use locally with `app.py`.

In [ ]:
# Zip and download (works on Kaggle/Colab)
import shutil
shutil.make_archive("accent-classifier-final", "zip", FINAL_MODEL_DIR)

# On Colab:
# from google.colab import files
# files.download("accent-classifier-final.zip")

print("Model zipped! Download 'accent-classifier-final.zip' from the file browser.")